# Construction de la première table analytique

## Objectif

Cette étape consiste à réunir les principales tables du dataset Olist dans une table analytique exploitable.

Le notebook couvre :

- les commandes ;
- les articles ;
- les clients ;
- les produits ;
- les catégories traduites ;
- les paiements ;
- les avis ;
- les vendeurs ;
- la géolocalisation ;
- les premiers KPI métier.

> Cette étape ne remplace pas encore le nettoyage final dans `src/cleaning.py`.
> Elle sert à comprendre les relations entre les données et à construire un premier modèle analytique.

## Notions Data importantes

### Granularité

La granularité décrit ce que représente une ligne.

Exemples :

- `orders` : une ligne par commande ;
- `order_items` : une ligne par article dans une commande ;
- `payments` : une ligne par paiement ;
- `reviews` : une ligne par avis.

### Clé primaire

Une clé primaire identifie une ligne de manière unique.

Exemple :

- `order_id` dans `orders`.

### Clé étrangère

Une clé étrangère relie une table à une autre.

Exemple :

- `customer_id` relie `orders` à `customers`.

### Risque de jointure many-to-many

Si deux tables contiennent plusieurs lignes pour une même clé, une jointure directe peut multiplier artificiellement les lignes.

Exemple :

- 3 articles ;
- 2 paiements ;
- jointure directe : 6 lignes.

Pour éviter cela, les paiements et les avis sont agrégés au niveau de la commande avant la jointure.

## 1. Imports et configuration

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Chemins du projet

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print(f"Racine du projet : {PROJECT_ROOT}")
print(f"Données brutes : {RAW_DATA_DIR}")

## 3. Chargement des tables

Chaque fichier CSV est chargé séparément afin de conserver la logique relationnelle du dataset.

In [ ]:
orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
products = pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv")
category_translation = pd.read_csv(
    RAW_DATA_DIR / "product_category_name_translation.csv"
)
payments = pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv")

datasets = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "products": products,
    "category_translation": category_translation,
    "payments": payments,
    "reviews": reviews,
    "sellers": sellers,
    "geolocation": geolocation,
}

pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
        }
        for name, dataframe in datasets.items()
    ]
)

## 4. Vérification des granularités principales

Cette vérification permet de confirmer les clés et d'éviter les mauvaises jointures.

In [ ]:
granularity_checks = pd.DataFrame(
    [
        {
            "table": "orders",
            "rows": len(orders),
            "key_unique": orders["order_id"].is_unique,
            "unique_keys": orders["order_id"].nunique(),
        },
        {
            "table": "customers",
            "rows": len(customers),
            "key_unique": customers["customer_id"].is_unique,
            "unique_keys": customers["customer_id"].nunique(),
        },
        {
            "table": "products",
            "rows": len(products),
            "key_unique": products["product_id"].is_unique,
            "unique_keys": products["product_id"].nunique(),
        },
        {
            "table": "sellers",
            "rows": len(sellers),
            "key_unique": sellers["seller_id"].is_unique,
            "unique_keys": sellers["seller_id"].nunique(),
        },
    ]
)

granularity_checks

## 5. Construction de la base `sales`

La table `order_items` est utilisée comme point de départ.

Sa granularité est :

**1 ligne = 1 article dans une commande**

La première jointure ajoute les informations de la commande.

In [ ]:
sales = order_items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one",
)

print(f"sales après orders : {sales.shape}")
sales.head()

## 6. Conversion des dates

Les dates sont converties en type `datetime`.

Cela permet :

- les regroupements mensuels ;
- les calculs de délai ;
- les analyses chronologiques.

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    sales[column] = pd.to_datetime(
        sales[column],
        errors="coerce",
    )

## 7. Feature engineering temporel

Le feature engineering consiste à créer de nouvelles variables à partir des données existantes.

In [ ]:
sales["purchase_year"] = sales["order_purchase_timestamp"].dt.year
sales["purchase_quarter"] = sales["order_purchase_timestamp"].dt.quarter
sales["purchase_month"] = sales["order_purchase_timestamp"].dt.month
sales["purchase_year_month"] = (
    sales["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype("string")
)
sales["purchase_day_of_week"] = (
    sales["order_purchase_timestamp"].dt.day_name()
)
sales["purchase_hour"] = sales["order_purchase_timestamp"].dt.hour

## 8. Création des montants et délais

Le dataset ne contient pas le coût d'achat réel des produits.

On peut donc calculer :

- le chiffre d'affaires produit ;
- les frais de livraison ;
- la valeur totale ;
- les délais de livraison.

On ne peut pas encore calculer une marge réelle.

In [ ]:
sales["product_revenue"] = sales["price"]
sales["total_item_value"] = (
    sales["price"] + sales["freight_value"]
)

sales["delivery_time_days"] = (
    sales["order_delivered_customer_date"]
    - sales["order_purchase_timestamp"]
).dt.days

sales["delivery_lateness_days"] = (
    sales["order_delivered_customer_date"]
    - sales["order_estimated_delivery_date"]
).dt.days

sales["is_late_delivery"] = (
    sales["delivery_lateness_days"] > 0
)

## 9. Ajout des clients

`customer_id` relie la commande à une ligne client.

`customer_unique_id` permet de reconnaître le même client sur plusieurs commandes.

In [ ]:
sales = sales.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

print(f"sales après customers : {sales.shape}")

## 10. Enrichissement des produits

Les catégories sont d'abord traduites en anglais.

Une catégorie finale est ensuite créée avec cette règle :

1. catégorie anglaise ;
2. sinon catégorie portugaise ;
3. sinon `unknown`.

In [ ]:
products_enriched = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

products_enriched["product_category"] = (
    products_enriched["product_category_name_english"]
    .fillna(products_enriched["product_category_name"])
    .fillna("unknown")
)

In [ ]:
sales = sales.merge(
    products_enriched[
        [
            "product_id",
            "product_category",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one",
)

print(f"sales après products : {sales.shape}")

## 11. Préparation des paiements

La table `payments` peut contenir plusieurs lignes par commande.

Elle est donc agrégée avant la jointure.

In [ ]:
payment_summary = (
    payments.groupby("order_id", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_type_count=("payment_type", "nunique"),
        max_payment_installments=("payment_installments", "max"),
    )
)

main_payment_type = (
    payments.sort_values(
        ["order_id", "payment_value"],
        ascending=[True, False],
    )
    .drop_duplicates(subset="order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)

payment_summary = payment_summary.merge(
    main_payment_type,
    on="order_id",
    how="left",
    validate="one_to_one",
)

payment_summary.head()

In [ ]:
sales = sales.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="many_to_one",
)

print(f"sales après payments : {sales.shape}")

## 12. Préparation des avis clients

Certaines commandes peuvent avoir plusieurs lignes d'avis.

Les avis sont agrégés au niveau de la commande avant la jointure.

In [ ]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce",
    )

review_summary = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count"),
        has_review_comment=(
            "review_comment_message",
            lambda values: values.notna().any(),
        ),
        review_creation_date=("review_creation_date", "min"),
        review_answer_timestamp=("review_answer_timestamp", "max"),
    )
)

review_summary.head()

In [ ]:
sales = sales.merge(
    review_summary,
    on="order_id",
    how="left",
    validate="many_to_one",
)

sales["customer_satisfaction"] = pd.cut(
    sales["review_score"],
    bins=[0, 2, 3, 5],
    labels=[
        "dissatisfied",
        "neutral",
        "satisfied",
    ],
)

print(f"sales après reviews : {sales.shape}")

## 13. Ajout des vendeurs

Chaque article est associé à un vendeur grâce à `seller_id`.

In [ ]:
sales = sales.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one",
)

print(f"sales après sellers : {sales.shape}")

## 14. Préparation de la géolocalisation

La table `geolocation` contient plusieurs lignes pour un même préfixe postal.

Pour éviter une jointure many-to-many, on agrège les coordonnées par préfixe postal.

In [ ]:
geolocation_summary = (
    geolocation.groupby(
        "geolocation_zip_code_prefix",
        as_index=False,
    )
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean"),
        geo_city=("geolocation_city", "first"),
        geo_state=("geolocation_state", "first"),
    )
)

geolocation_summary.head()

## 15. Ajout de la géolocalisation client

La jointure utilise le préfixe postal du client.

In [ ]:
customer_geolocation = geolocation_summary.rename(
    columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "latitude": "customer_latitude",
        "longitude": "customer_longitude",
        "geo_city": "customer_geo_city",
        "geo_state": "customer_geo_state",
    }
)

sales = sales.merge(
    customer_geolocation,
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one",
)

print(f"sales après géolocalisation client : {sales.shape}")

## 16. Ajout de la géolocalisation vendeur

In [ ]:
seller_geolocation = geolocation_summary.rename(
    columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "latitude": "seller_latitude",
        "longitude": "seller_longitude",
        "geo_city": "seller_geo_city",
        "geo_state": "seller_geo_state",
    }
)

sales = sales.merge(
    seller_geolocation,
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one",
)

print(f"sales après géolocalisation vendeur : {sales.shape}")

## 17. Contrôle de la table analytique

La granularité finale reste :

**1 ligne = 1 article dans une commande**

Les informations de commande, client, produit, paiement, avis et vendeur sont répétées sur chaque ligne article.

In [ ]:
sales_checks = pd.Series(
    {
        "nombre_lignes": len(sales),
        "commandes_uniques": sales["order_id"].nunique(),
        "articles_uniques": sales[
            ["order_id", "order_item_id"]
        ].drop_duplicates().shape[0],
        "clients_non_trouves": sales["customer_unique_id"].isna().sum(),
        "produits_non_trouves": sales["product_category"].isna().sum(),
        "paiements_non_trouves": sales["total_payment_value"].isna().sum(),
        "avis_non_trouves": sales["review_score"].isna().sum(),
        "vendeurs_non_trouves": sales["seller_state"].isna().sum(),
    }
)

sales_checks

## 18. Périmètre métier

Pour les premiers KPI, on retient uniquement les commandes livrées.

Cela évite de compter comme chiffre d'affaires final des commandes annulées ou indisponibles.

In [ ]:
delivered_sales = sales.loc[
    sales["order_status"].eq("delivered")
].copy()

print(f"Articles toutes commandes : {len(sales):,}")
print(f"Articles livrés : {len(delivered_sales):,}")

## 19. KPI globaux

Attention :

- les montants article peuvent être additionnés directement ;
- le montant total payé doit être calculé au niveau de la commande pour éviter les doubles comptages.

In [ ]:
order_level = (
    delivered_sales[
        [
            "order_id",
            "customer_unique_id",
            "total_payment_value",
            "review_score",
            "is_late_delivery",
            "delivery_time_days",
        ]
    ]
    .drop_duplicates(subset="order_id")
)

global_kpis = pd.Series(
    {
        "commandes_livrees": delivered_sales["order_id"].nunique(),
        "articles_vendus": len(delivered_sales),
        "clients_uniques": delivered_sales["customer_unique_id"].nunique(),
        "chiffre_affaires_produits": delivered_sales["price"].sum(),
        "frais_livraison": delivered_sales["freight_value"].sum(),
        "valeur_totale_articles": delivered_sales["total_item_value"].sum(),
        "montant_total_paye": order_level["total_payment_value"].sum(),
        "panier_moyen": order_level["total_payment_value"].mean(),
        "note_moyenne": order_level["review_score"].mean(),
        "taux_retard": order_level["is_late_delivery"].mean() * 100,
    }
)

global_kpis

## 20. Ventes mensuelles

In [ ]:
monthly_sales = (
    delivered_sales.groupby("purchase_year_month", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
        items=("order_item_id", "count"),
        product_revenue=("price", "sum"),
        freight_revenue=("freight_value", "sum"),
        total_revenue=("total_item_value", "sum"),
    )
    .sort_values("purchase_year_month")
)

monthly_sales["average_order_value"] = (
    monthly_sales["total_revenue"]
    / monthly_sales["orders"]
)

monthly_sales.head()

## 21. Performance par catégorie produit

In [ ]:
category_performance = (
    delivered_sales.groupby("product_category", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        customers=("customer_unique_id", "nunique"),
        product_revenue=("price", "sum"),
        freight_revenue=("freight_value", "sum"),
        average_review_score=("review_score", "mean"),
    )
    .sort_values("product_revenue", ascending=False)
)

category_performance.head(10)

## 22. Performance par État client

In [ ]:
customer_state_performance = (
    delivered_sales.groupby("customer_state", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        orders=("order_id", "nunique"),
        items=("order_item_id", "count"),
        total_revenue=("total_item_value", "sum"),
        average_delivery_time=("delivery_time_days", "mean"),
        average_review_score=("review_score", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

customer_state_performance.head(10)

## 23. Performance des vendeurs

In [ ]:
seller_performance = (
    delivered_sales.groupby("seller_id", as_index=False)
    .agg(
        seller_city=("seller_city", "first"),
        seller_state=("seller_state", "first"),
        orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        product_revenue=("price", "sum"),
        average_delivery_time=("delivery_time_days", "mean"),
        late_delivery_rate=("is_late_delivery", "mean"),
        average_review_score=("review_score", "mean"),
    )
)

seller_performance["late_delivery_rate"] *= 100

seller_performance.sort_values(
    "product_revenue",
    ascending=False,
).head(10)

## 24. Satisfaction et retard

Cette analyse permet de répondre à la question :

**Les retards diminuent-ils la satisfaction client ?**

In [ ]:
order_review_analysis = (
    delivered_sales[
        [
            "order_id",
            "review_score",
            "customer_satisfaction",
            "is_late_delivery",
            "delivery_time_days",
            "delivery_lateness_days",
        ]
    ]
    .drop_duplicates(subset="order_id")
)

review_by_lateness = (
    order_review_analysis.groupby(
        "is_late_delivery",
        as_index=False,
    )
    .agg(
        orders=("order_id", "nunique"),
        average_review_score=("review_score", "mean"),
        average_delivery_time=("delivery_time_days", "mean"),
        average_lateness=("delivery_lateness_days", "mean"),
    )
)

review_by_lateness

## 25. Moyens de paiement

In [ ]:
order_payment_analysis = (
    delivered_sales[
        [
            "order_id",
            "main_payment_type",
            "total_payment_value",
            "max_payment_installments",
        ]
    ]
    .drop_duplicates(subset="order_id")
)

payment_type_performance = (
    order_payment_analysis.groupby(
        "main_payment_type",
        as_index=False,
    )
    .agg(
        orders=("order_id", "nunique"),
        total_payment_value=("total_payment_value", "sum"),
        average_payment_value=("total_payment_value", "mean"),
        average_installments=("max_payment_installments", "mean"),
    )
    .sort_values("orders", ascending=False)
)

payment_type_performance

## 26. Colonnes principales de la table finale

In [ ]:
selected_columns = [
    "order_id",
    "order_item_id",
    "order_status",
    "order_purchase_timestamp",
    "purchase_year_month",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "product_id",
    "product_category",
    "seller_id",
    "seller_city",
    "seller_state",
    "price",
    "freight_value",
    "total_item_value",
    "total_payment_value",
    "main_payment_type",
    "max_payment_installments",
    "review_score",
    "customer_satisfaction",
    "delivery_time_days",
    "delivery_lateness_days",
    "is_late_delivery",
    "customer_latitude",
    "customer_longitude",
    "seller_latitude",
    "seller_longitude",
]

sales[selected_columns].head()

## Conclusion de l'étape 4

La première table analytique est maintenant construite.

Elle contient :

- les commandes ;
- les articles ;
- les clients ;
- les produits ;
- les catégories ;
- les paiements ;
- les avis ;
- les vendeurs ;
- la géolocalisation ;
- les variables temporelles ;
- les délais ;
- les premiers KPI.

## Limite importante

Cette table reste une table d'exploration.

La prochaine étape sera de déplacer les règles de transformation dans :

- `src/cleaning.py` ;
- `src/validation.py`.

Ensuite, les données nettoyées seront enregistrées dans `data/processed/`.